# Lenguajes y Autómatas II (SCD-1016)
## Sesión 32 · Asignación Ingenua de Registros (Naive Allocation)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Dónde vive cada temporal mientras el backend lo está usando?
>
> **Objetivo de la sesión:** Asignar cada temporal a una ranura fija de la pila y elegir los registros de trabajo con los que el código generado carga, opera y almacena.

**Entregable:** tu implementación de slot_temporal() y registro_de_trabajo() en artesano_sesion32_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion32.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes el nombre de un temporal (t0, t1, ...) o el índice de un registro de trabajo, y devuelves su desplazamiento en el marco o el nombre AT&T del registro que le toca.

Al final de la sesión tendrás: **Un mapa de temporales a desplazamientos de pila y una tabla de registros de trabajo, implementados y verificados caso por caso.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 32](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion32.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion32.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion32.h"

int slot_temporal(const char *temporal);
const char *registro_de_trabajo(int indice);
```

- Cada temporal tN ocupa 8 bytes: su desplazamiento es -8 * (N + 1).
- Los registros de trabajo son %rax, %rbx, %r10 y %r11, en ese orden; del índice 4 en adelante no hay registro.
- Un nombre que no sea t seguido de dígitos se considera inválido y vale cero.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion32_c.c` junto con `artesano_sesion32_c.c`
> (o `ia_sesion32_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `slot_temporal("t0")` | -8 |
| 2 | `slot_temporal("t7")` | -64 |
| 3 | `slot_temporal("x")` | 0 |
| 4 | `registro_de_trabajo(0)` | `%rax` |
| 5 | `registro_de_trabajo(3)` | `%r11` |
| 6 | `registro_de_trabajo(4)` | `ninguno` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 32 - Asignador directo de temporales a registros ===
Caso                               esperado:              obtenido:              resultado
[ 1] slot_temporal("t0")                esperado: -8           obtenido: -8           OK
[ 2] slot_temporal("t7")                esperado: -64          obtenido: -64          OK
[ 3] slot_temporal("x")                 esperado: 0            obtenido: 0            OK
[ 4] registro_de_trabajo(0)             esperado: %rax         obtenido: %rax         OK
[ 5] registro_de_trabajo(3)             esperado: %r11         obtenido: %r11         OK
[ 6] registro_de_trabajo(4)             esperado: ninguno      obtenido: ninguno      OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion32.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] slot_temporal acierta el desplazamiento de cada temporal y rechaza nombres mal formados con cero.
- [ ] registro_de_trabajo respeta el orden del pool y responde ninguno cuando el índice se sale.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion32_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion32.h
/* api_sesion32.h - contrato de la Sesion 32
 * Asignador directo de temporales a registros
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION32_H
#define API_SESION32_H

int slot_temporal(const char *temporal);
const char *registro_de_trabajo(int indice);

#endif /* API_SESION32_H */


In [ ]:
%%writefile artesano_sesion32_c.c
/* artesano_sesion32_c.c - Fase 1: El Artesano (sin IA)
 * Asignador directo de temporales a registros.
 *
 * Implementa lo que declara api_sesion32.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion32.h"
#include <stdio.h>
#include <string.h>

int slot_temporal(const char *temporal) {
    (void)temporal; /* TODO: t0 -> -8, t1 -> -16, ... */
    return 0;
}

const char *registro_de_trabajo(int indice) {
    (void)indice; /* TODO: 0 -> %rax, 1 -> %rbx, ... */
    return "ninguno";
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion32_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion32_c.c
/* ia_sesion32_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion32.h. No puede traer main().
 */
#include "api_sesion32.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion32_c.c
/* harness_sesion32_c.c - auditoria automatica de la Sesion 32
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion32_c.c artesano_sesion32_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion32.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 32 - Asignador directo de temporales a registros ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = slot_temporal("t0"); verificar_int("slot_temporal(\"t0\")", (long)(-8), (long)(r)); }
    { int r = slot_temporal("t7"); verificar_int("slot_temporal(\"t7\")", (long)(-64), (long)(r)); }
    { int r = slot_temporal("x"); verificar_int("slot_temporal(\"x\")", (long)(0), (long)(r)); }
    { const char *r = registro_de_trabajo(0); verificar_str("registro_de_trabajo(0)", "%rax", r); }
    { const char *r = registro_de_trabajo(3); verificar_str("registro_de_trabajo(3)", "%r11", r); }
    { const char *r = registro_de_trabajo(4); verificar_str("registro_de_trabajo(4)", "ninguno", r); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion32.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion32_c.c artesano_sesion32_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion32_c.c ia_sesion32_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion32.md` a `auditorias/auditoria_sesion32.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
